# Propensity score matching: the NSW experiment and its replications

Can a study without a randomised control group find the effect that a randomised experiment found? LaLonde (1986) took the National Supported Work (NSW) experiment, whose controls give the effect of the training on the 1978 earnings of the men treated, about $1,794, and found that nonexperimental estimates from the PSID and CPS surveys missed it widely. Dehejia and Wahba (1999) matched each treated man to the survey men with the nearest propensity score and came close. Smith and Todd (2005) replied that this works only in Dehejia and Wahba's subsample and for some specifications; Dehejia (2005) replied that the specification must be chosen for each sample by the balance of the covariates. Their tables are printed beside Julia's, and the matches were checked in R against MatchIt, Matching and cobalt.

The data are LaLonde's NSW men and the PSID and CPS comparison groups as Dehejia distributes them on the NBER data page (users.nber.org/~rdehejia/data). They are not offered for redistribution with Pollis, so they are read into memory and not saved. The estimators come from the PropensityMatching package, of Compose › New Methods: no registered Julia package does propensity score matching as these papers do.

In [ ]:
Pollis.packages("PropensityMatching" => "https://github.com/Trumpingtons/PropensityMatching.jl", "Plots")

## Set-up

The ten files of Dehejia's data page, read into named tuples of columns: LaLonde's NSW treated and controls (with no 1974 earnings), the RE74 subset that Dehejia and Wahba use (`treated`, `controls`), and the three PSID and three CPS comparison groups; `using PropensityMatching` for the estimators, with the papers' samples and specifications from it (`stack`, `thousands`, `dw2002`, `dehejia2005`, `stscore`, `dollars`); and `replicate`, which prints each value in full, rounded to the published precision, beside the published value, and whether they agree to every printed digit.

In [ ]:
using PropensityMatching, Plots, Printf, Statistics, DelimitedFiles, Downloads
using PropensityMatching: stack, thousands, dw2002, dehejia2005, stscore, dollars

"""
	replicate(title, rows)

Print each Julia value in full, rounded to the published precision and beside the published value, with their agreement.
Each row is `(label, value, published)`, with `published` written as in the source: its decimals give the published
precision (in scientific notation, as 1.64e-09, those of the mantissa).
"""
function replicate(title, rows)
	w = maximum(length(first(r)) for r in rows) + 2
	println("\n", title, "\n", rpad("", w), lpad("Julia", 18), lpad("rounded", 14), lpad("published", 14))
	for (label, value, published) in rows
		p = parse(Float64, replace(published, "," => ""))
		mantissa, exponent = occursin(r"[eE]", published) ? split(published, r"[eE]") : (published, "0")
		d = (occursin('.', mantissa) ? length(split(mantissa, '.')[2]) : 0) - parse(Int, exponent)
		r = round(value, digits = d)
		agreement = r == p ? "every printed digit" : abs(r - p) <= 1.5 * 10.0^-d ? "one in the last digit" : "differs by $(round(r - p, digits = d))"
		rounded = d <= 0 ? string(Int(r)) : d <= 6 ? Printf.format(Printf.Format("%.$(d)f"), r) : string(r)
		println(rpad(label, w), lpad(@sprintf("%.8g", value), 18), lpad(rounded, 14), lpad(published, 14), "  ", agreement)
	end
end

"""Read one of Dehejia's files into a named tuple of columns, in memory only."""
function nber(file)
	m = readdlm(IOBuffer(take!(Downloads.download("https://users.nber.org/~rdehejia/data/$file.txt", IOBuffer()))))
	names = size(m, 2) == 10 ? (:treat, :age, :educ, :black, :hisp, :married, :nodegree, :re74, :re75, :re78) :
		(:treat, :age, :educ, :black, :hisp, :married, :nodegree, :re75, :re78)
	NamedTuple{names}(Tuple(Float64.(m[:, j]) for j in 1:size(m, 2)))
end

lalondetreated, lalondecontrols = nber("nsw_treated"), nber("nsw_control")
treated, controls = nber("nswre74_treated"), nber("nswre74_control")
groups = ["PSID-1" => nber("psid_controls"), "PSID-2" => nber("psid2_controls"), "PSID-3" => nber("psid3_controls"),
	"CPS-1" => nber("cps_controls"), "CPS-2" => nber("cps2_controls"), "CPS-3" => nber("cps3_controls")]
psid, cps = groups[1][2], groups[4][2]

## 1. The experiment and the naive comparisons

The experiment's answer is the difference between the treated and the randomised controls. Replacing the controls by a survey sample gives very different answers, because the survey men are older, more educated and earned far more before the programme (Dehejia and Wahba, Tables 1 and 3, column 1). The PSID-3 share of Hispanics is printed .18; the data give .117, and the other means agree, so it is shown but not compared.

In [ ]:
names = ["Age", "Education", "Black", "Hispanic", "No degree", "Married", "RE74", "RE75"]
vars = (:age, :educ, :black, :hisp, :nodegree, :married, :re74, :re75)
table1 = [
	"NSW/Lalonde, treated" => (lalondetreated, ["24.63", "10.38", ".80", ".09", ".73", ".17", "", "3066"]),
	"NSW/Lalonde, control" => (lalondecontrols, ["24.45", "10.19", ".80", ".11", ".81", ".16", "", "3026"]),
	"RE74 subset, treated" => (treated, ["25.81", "10.35", ".84", ".059", ".71", ".19", "2096", "1532"]),
	"RE74 subset, control" => (controls, ["25.05", "10.09", ".83", ".1", ".83", ".15", "2107", "1267"]),
	"PSID-1" => (groups[1][2], ["34.85", "12.11", ".25", ".032", ".31", ".87", "19429", "19063"]),
	"PSID-2" => (groups[2][2], ["36.10", "10.77", ".39", ".067", ".49", ".74", "11027", "7569"]),
	"PSID-3" => (groups[3][2], ["38.25", "10.30", ".45", "", ".51", ".70", "5566", "2611"]),
	"CPS-1" => (groups[4][2], ["33.22", "12.02", ".07", ".07", ".29", ".71", "14016", "13650"]),
	"CPS-2" => (groups[5][2], ["28.25", "11.24", ".11", ".08", ".45", ".46", "8728", "7397"]),
	"CPS-3" => (groups[6][2], ["28.03", "10.23", ".21", ".14", ".60", ".51", "5619", "2467"]),
]
for (title, (s, published)) in table1
	replicate("Table 1, " * title, [(n, mean(s[v]), p) for (n, v, p) in zip(names, vars, published) if haskey(s, v) && p != ""])
end
@printf("\nPSID-3, Hispanic: %.4f, printed .18\n", mean(groups[3][2].hisp))

replicate("Table 3, column (1): unadjusted difference", collect(zip(["NSW"; first.(groups)],
	[mean(treated.re78) - mean(c.re78) for c in [controls; last.(groups)]], ["1794", "-15205", "-3647", "1069", "-8498", "-3822", "-635"])))

## 2. The propensity score

The probability of treatment given the covariates, fitted by the package's `logit` by Newton-Raphson, with earnings in thousands of dollars. Smith and Todd's Table 3 prints Dehejia and Wahba's (2002) specifications fitted to the NSW treated and controls together against each survey sample: the coefficients of earnings per dollar and of squared earnings per squared dollar.

In [ ]:
b, se = dollars(stscore(treated, controls, cps, :cps)[3], :cps)
published = [
	("Age", "2.7441", "0.2681"),
	("Age squared", "-0.0779", "0.0085"),
	("Age cubed/1000", "0.6769", "0.0837"),
	("Years of schooling", "1.2274", "0.2249"),
	("Years of schooling squared", "-0.0692", "0.0120"),
	("High school dropout", "1.3515", "0.2588"),
	("Married", "-1.7307", "0.1932"),
	("Black", "3.9988", "0.2000"),
	("Hispanic", "2.2457", "0.2637"),
	("Real earnings in 1974", "-0.00007", "0.00007"),
	("Real earnings in 1975", "-0.00020", "0.00003"),
	("Zero earnings in 1974", "1.9368", "0.2209"),
	("Zero earnings in 1975", "0.2513", "0.1994"),
	("Schooling x real earnings in 1974", "0.00001", "6.14e-06"),
	("Intercept", "-39.8326", "3.0398"),
]
replicate("Smith and Todd, Table 3, DW sample, CPS: coefficients", [(l, v, p) for ((l, p, _), v) in zip(published, b)])
replicate("Smith and Todd, Table 3, DW sample, CPS: standard errors", [(l, v, p) for ((l, _, p), v) in zip(published, se)])

b, se = dollars(stscore(treated, controls, psid, :psid)[3], :psid)
published = [
	("Age", "0.2386"),
	("Age squared", "-0.0051"),
	("Years of schooling", "0.9748"),
	("Years of schooling squared", "-0.0525"),
	("High school dropout", "0.7490"),
	("Married", "-2.0301"),
	("Black", "2.6277"),
	("Hispanic", "3.3643"),
	("Real earnings in 1974", "-0.00002"),
	("Real earnings in 1974 squared", "1.64e-09"),
	("Real earnings in 1975", "-0.00025"),
	("Real earnings in 1975 squared", "5.28e-10"),
	("Zero earnings in 1974", "3.2583"),
	("Zero earnings in 1975", "-1.0396"),
	("Zero earnings in 1974 x Hispanic", "-1.4627"),
	("Intercept", "-8.5683"),
]
replicate("Smith and Todd, Table 3, DW sample, PSID: coefficients", [(l, v, p) for ((l, p), v) in zip(published, b)])

## 3. Matching

Each treated man is matched, with replacement, to the comparison man with the nearest score, and the effect on the treated is the mean difference in 1978 earnings. When several comparison men are exactly as near, the first in the data is taken; `tierange` gives the smallest and largest effect over every way of breaking those ties. Dehejia (2005, Table 2) chose a specification for each sample by balance; his Tables 3 and 4 match on each. The DW sample is the RE74 subset; LaLonde's own sample has no 1974 earnings, so only the Lalonde specifications fit it.

In [ ]:
"""The sample, its score and the nearest-neighbour effect of a specification."""
function dehejia(t, comparison, name)
	s = stack(t, comparison)
	score = logit(dehejia2005(thousands(s), name), s.treat).fitted
	s, score, att(s.re78, nnmatch(score, s.treat)...)
end

cases = [(treated, cps, "CPS/Lalonde"), (treated, cps, "CPS/ST"), (treated, psid, "PSID/DW"), (treated, psid, "PSID/Lalonde"),
	(treated, psid, "PSID/ST"), (lalondetreated, cps, "CPS/Lalonde"), (lalondetreated, psid, "PSID/Lalonde")]
labels = ["DW sample, CPS, CPS/Lalonde specification (Table 4)", "DW sample, CPS, CPS/Smith-Todd specification (Table 4)", "DW sample, PSID, PSID/DW specification (Table 3)", "DW sample, PSID, PSID/Lalonde specification (Table 4)", "DW sample, PSID, PSID/Smith-Todd specification (Table 4)", "Lalonde sample, CPS, CPS/Lalonde specification (Table 3)", "Lalonde sample, PSID, PSID/Lalonde specification (Table 3)"]
replicate("Dehejia (2005), nearest neighbour", collect(zip(labels, [dehejia(c...)[3] for c in cases], ["1867", "1283", "1869", "1313", "1991", "880", "863"])))

In Dehejia's CPS/DW specification 22 treated men have two or more CPS men at exactly the same score. The printed 1,589 (Table 3) is inside the range over ties, but no rule a reader could follow gives it.

In [ ]:
s, score, effect = dehejia(treated, cps, "CPS/DW")
tied = count(length(m) > 1 for m in last(nnmatch(score, s.treat; ties = :all)))
@printf("CPS/DW: first of ties %.0f, range over ties %.0f to %.0f, %d treated with ties; printed 1,589\n", effect, tierange(s.re78, score, s.treat)..., tied)

## 4. Balance

Matching works if the matched comparison men look like the treated. The standardised difference of each covariate, the difference in means over the treated standard deviation (left raw for a binary covariate, as R's cobalt reports it), before and after matching, for Dehejia's PSID/DW specification. cobalt's `bal.tab`, given these weights, prints the same 16 values to 4 decimals.

In [ ]:
s, score, effect = dehejia(treated, psid, "PSID/DW")
t, matches = nnmatch(score, s.treat)
X = hcat(s.age, s.educ, s.black, s.hisp, s.married, s.nodegree, s.re74, s.re75)
covariates = ["age", "educ", "black", "hisp", "married", "nodegree", "re74", "re75"]
before, after = balance(X, s.treat), balance(X, s.treat; weights = matchweights(length(s.treat), t, matches))
@printf("Effect on the treated: %.2f (experiment: 1,794)\n", effect)
foreach(((c, b, a),) -> @printf("%-10s%10.4f%10.4f\n", c, b, a), zip(covariates, before, after))
scatter(abs.(before), 1:8, label = "before matching", yticks = (1:8, covariates), xlabel = "absolute standardised difference", size = (560, 330))
scatter!(abs.(after), 1:8, label = "after matching")
vline!([0.1], label = "0.1", linestyle = :dash)

## 5. Smith and Todd's test: the bias

Smith and Todd put the NSW controls in place of the treated: the training did not touch them, so any effect an estimator finds is its bias. Table 5, column (1) is the mean difference and Table 7, column (3) the regression on Dehejia and Wahba's (2002) covariates. Their Table 7 prints the PSID mean difference as -17,796 (846): that is the gap in 1975 earnings (their Table 8); the gap in 1978 earnings is the -16,999 of Table 5.

In [ ]:
for (name, comparison, kind, published) in [("CPS", cps, :cps, ["-10291", "-34", "486"]), ("PSID", psid, :psid, ["-16999", "-1285", "960"])]
	nsw = stack(controls, comparison)
	d = [ones(length(controls.age)); zeros(length(comparison.age))]
	coefficients, errors = ols(nsw.re78, hcat(d, dw2002(thousands(nsw), kind)))
	replicate("Smith and Todd, bias, " * name, collect(zip(["Mean difference, Table 5 (1)", "Regression, Table 7 (3)", "Standard error, Table 7 (3)"],
		[mean(nsw.re78[d .== 1]) - mean(nsw.re78[d .== 0]), coefficients[2], errors[2]], published)))
end
@printf("\nPSID gap in 1975 earnings: %.2f, printed in Table 7 as the mean difference -17,796\n", mean(controls.re75) - mean(psid.re75))

## 6. Local linear matching on the trimmed support

Smith and Todd's preferred estimator matches on the log-odds of the score, log(p/(1 - p)), and replaces each match by a local linear regression of the comparison outcomes on the score, with a quartic kernel (their eq. 11). They first drop the treated where the density of the comparison scores is thinnest, 2 per cent of them, with Silverman's bandwidth (Section 3.3 and footnote 20).

In [ ]:
s, d, m = stscore(treated, controls, cps, :cps)
keep = [i > length(treated.age) for i in eachindex(d)]  # the NSW controls in place of the treated
z, y, dc = logodds(m.fitted)[keep], s.re78[keep], d[keep]
support = trimmedsupport(z, dc)
println("Trimmed common support: ", length(support), " of ", count(dc .== 1), " NSW controls")
replicate("Smith and Todd, Table 5, CPS, local linear", collect(zip(["Bandwidth 1 (6)", "Bandwidth 4 (7)"],
	[locallinear(y, z, dc, h; treated = support) for h in (1, 4)], ["-88", "-67"])))
@printf("\nSingle nearest neighbour (2): %.0f, over ties %.0f to %.0f, printed 407\n", att(y, nnmatch(z, dc)...), tierange(y, z, dc)...)

## 7. The branch: matching or regression?

Smith and Todd's question is whether matching removes more bias than a regression on the same covariates. With the NSW controls in place of the treated, the estimator with the smaller absolute bias wins. If local linear matching (bandwidth 4) has the smaller bias for both survey samples, `matching` is true; otherwise regression does as well or better on at least one, and neither method can be trusted to recover the experiment on its own. The PSID local linear estimate does not reproduce Smith and Todd's printed -122 and 143 (see What differs), so its value is Julia's.

In [ ]:
bias = map([("CPS", cps, :cps), ("PSID", psid, :psid)]) do (name, comparison, kind)
	s, d, m = stscore(treated, controls, comparison, kind)
	keep = [i > length(treated.age) for i in eachindex(d)]
	z, y, dc = logodds(m.fitted)[keep], s.re78[keep], d[keep]
	local_linear = locallinear(y, z, dc, 4; treated = trimmedsupport(z, dc))
	r = stack(controls, comparison)
	dr = [ones(length(controls.age)); zeros(length(comparison.age))]
	regression = ols(r.re78, hcat(dr, dw2002(thousands(r), kind)))[1][2]
	@printf("%-6s local linear %10.2f   regression %10.2f\n", name, local_linear, regression)
	abs(local_linear) < abs(regression)
end
matching = all(bias)

## 8. What differs, and why

Dehejia and Wahba (1999), Table 1: 62 of 77 means agree to every printed digit and 15 in the last, most of them truncated (14016.8 printed 14016). Table 3, column (1): 6 of 7, PSID-3 1,069.85 printed 1,069. Smith and Todd's Table 3: all 16 PSID coefficients and 29 of 30 CPS values; the CPS intercept is -39.832651, printed -39.8326. Tables 5 and 7: 5 of 6, the CPS mean difference -10,291.86 printed -10,291; local linear matching -87.0 and -67.94, printed -88 and -67. Dehejia (2005): 5 of 7, and 1,282.49 and 1,990.47 printed 1,283 and 1,991. The rest:

- **Ties decide the CPS estimates.** Dehejia's CPS/DW estimate runs from 1,055 to 1,964 over the ways of breaking exact ties (section 3); Smith and Todd's CPS single nearest neighbour, printed 407, from 41 to 828 (Julia's first of ties: 535). Neither paper says how ties were broken.
- **A printed mean difference is the 1975 gap.** Smith and Todd's Table 7 PSID mean difference, -17,796, is the gap in 1975 earnings; the 1978 gap is -16,999 (section 5).
- **A corrected mean.** Dehejia and Wahba's PSID-3 share of Hispanics, printed .18, is .117 in the data.
- **Dehejia and Wahba's own matches do not reproduce.** Their Table 3, columns (2) to (8): PSID-1 gives 560 against 1,691. Smith and Todd (footnote 40) note that the sample on which the score is fitted moves these estimates by $200-300.
- **Smith and Todd's other estimators.** Their PSID single nearest neighbour, 361, is what matching on p gives; on the log-odds they describe it is 319. Their ten-neighbour and trimmed estimators come within about $100, and the PSID local linear ones do not reproduce. Their PSID standard errors agree to about three significant figures.
- **Against R.** MatchIt, given these scores, picks one of the exactly nearest comparison men every time, and its effects equal Julia's where the tie choices coincide (1,869.07, 1,990.47, 862.72). Matching averages over near-ties within an internal tolerance, so it gives 1,829.82 for PSID/DW; on CPS/Lalonde it gives 1,858.87, Julia's average over ties.

**Now in Compose › New Methods.** Julia has no counterpart of MatchIt, Matching or cobalt. The New Methods package **PropensityMatching** has these estimators as documented functions (the logit score, both common supports, nearest-neighbour matching with its range over ties, local linear matching, the effect on the treated and the balance), tested against the three papers' tables. Find it in **Compose › New Methods**.

## References

- Dehejia, R. H. and Wahba, S. (1999). Causal effects in nonexperimental studies: reevaluating the evaluation of training programs. *Journal of the American Statistical Association* 94(448), 1053-1062. doi:10.1080/01621459.1999.10473858
- Dehejia, R. H. and Wahba, S. (2002). Propensity score-matching methods for nonexperimental causal studies. *Review of Economics and Statistics* 84(1), 151-161. doi:10.1162/003465302317331982
- Smith, J. A. and Todd, P. E. (2005). Does matching overcome LaLonde's critique of nonexperimental estimators? *Journal of Econometrics* 125(1-2), 305-353. doi:10.1016/j.jeconom.2004.04.011
- Dehejia, R. (2005). Practical propensity score matching: a reply to Smith and Todd. *Journal of Econometrics* 125(1-2), 355-364. doi:10.1016/j.jeconom.2004.04.012
- LaLonde, R. J. (1986). Evaluating the econometric evaluations of training programs with experimental data. *American Economic Review* 76(4), 604-620.